In [60]:
import pickle
import sys
import copy
import time
import os

import cobra
import sympy
import pandas as pd
import numpy as np

from tqdm import tqdm

sys.path.insert(1, '/home/hratch/Projects/human_me/scripts/')
from utils import functions as func
from utils import parameters as params

In [61]:
from utils.parameters import human_model as m_model
mu_val = 1e-9
n_cores = 10
base = 0
counter = 3

lp_path = '/data2/hratch/human_me/other/test_lp/'
with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
    tme_new = pickle.load(handle)

def add_sink(m, tme_new = None):
    '''m is a cobra.metabolite or metabolite ID'''
    
    if tme_new is None:
        with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
            tme_new = pickle.load(handle)
    
    if isinstance(m, cobra.Metabolite): # object
        m_id = m.id
    else: # string
        m_id = m
    
    tme_new.add_boundary(tme_new.metabolites.get_by_id(m_id), type ='sink')
    sln, stat, _ = tme_new.solve_lp(mu_val = mu_val)
    
    if not os.path.isfile('test_sinks.tab'):
        with open('test_sinks.tab', 'a+') as f:
            f.write('metabolite_id' + '\t' + 'status' + '\n')
        
        
    with open('test_sinks.tab', 'a+') as f:
        f.write(m_id + '\t' + str(stat.max()) + '\n')
        

def _add_sink(metabs_, tme_new = None):
    '''Metabs_ is a list of cobra.Metabolite or metabolite IDs'''
    if tme_new is None:
        with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
            tme_new = pickle.load(handle)
    if type(metabs_) != list:
        metabs_ = list(metabs_)

    for m in metabs_:
        if isinstance(m, cobra.Metabolite): # object
            tme_new.add_boundary(tme_new.metabolites.get_by_id(m.id), type ='sink')
        else: # string
            tme_new.add_boundary(tme_new.metabolites.get_by_id(m), type ='sink')
    
    sln, stat, _ = tme_new.solve_lp(mu_val = mu_val)
    return tme_new, sln, stat

In [17]:
# from macromolecules.macromolecule import Macromolecule
# from expression.build_me_model import flatten_list
# test_metab = list(m_model.metabolites)

# test = list()
# for m_id in flatten_list([[m.id.replace('_b', '_c') for m in list(r.metabolites)] for r in m_model.exchanges]):
#     try:
#         test.append(m_model.metabolites.get_by_id(m_id))
#     except: 
#         pass

# tme_sink, sln_sink, stat_sink = _add_sink(test_metab)


# with open(lp_path + 'working_version_' + str(counter) + '.pickle', 'rb') as handle:
#     tmef = pickle.load(handle)
# sln_f, stat_f, _ = tmef.solve_lp(mu_val = mu_val)

# max_val = max([abs(v) for v in tme_sink.infeasible_reactions(1e-9, sln_sink, stat_sink, tolerance = 0).values()])
# ifr = tmef.infeasible_reactions(1e-9, sln_f, stat_f, tolerance = max_val)
# get_metabs_of = [r for r in ifr if 'HGNC' not in r and 'biomass' not in r]

# metabs_ = list()
# metabs_ = list()
# for r in get_metabs_of:
#     metabs_ += [m.id for m in tmef.reactions.get_by_id(r).metabolites if not isinstance(m, Macromolecule)]
# metabs_ = list(set(metabs_))

metabs_ = ['lys_L_e', 'gln_L_e', 'glu_L_l', 'asp_L_c', 'glu_L_c', 'lys_L_c', 'arg_L_c', 'asp_L_l', 'adp_c', 
           'pi_c', 'glc_D_e', 'glc_D_c', 'adp_l', 'gsn_l', 'arg_L_l', 'atp_c', 'atp_l', 'gln_L_c', 'pi_l', 
           'gsn_c'] # adp_c gives 0

adp_c = tme_new.metabolites.get_by_id('adp_c')
test = flatten_list([[m.id for m in r.products] for r in tme_new.reactions if adp_c in r.reactants])
metabs_ = sorted(set(test).difference(metabs_))

dadp_c = tme_new.metabolites.get_by_id('dadp_c')
test = flatten_list([[m.id for m in r.products] for r in tme_new.reactions if adp_c in r.reactants])
test = sorted(set(test).difference(metabs_))

import multiprocessing
import gc

pool = multiprocessing.Pool(processes = n_cores)
try:
    res = pool.map(add_sink, test)
    pool.close()
    pool.join()
    gc.collect()
except:
    pool.close()
    pool.join()
    gc.collect()
    raise ValueError('Parallelization failed')

Getting MINOS parameters...
Done in 57.9268 seconds with status 1


In [97]:
test

['adp_l', 'atp_c']

In [95]:
test

['3pg_c',
 'adp_m',
 'adp_n',
 'adp_r',
 'adp_x',
 'amp_c',
 'cdp_c',
 'cmp_c',
 'dadp_c',
 'dag_hs_c',
 'dgmp_c',
 'gmp_c',
 'gtp_c',
 'h2o_c',
 'pyr_c',
 'trdox_c',
 'udp_c',
 'ump_c']

In [66]:
tme_new.pickle(lp_path + 'working_version_' + str(4) + '.pickle')

In [67]:
import pandas as pd
res_df = pd.DataFrame(data = {'reactions': [r.id for r in tme_new.reactions]})
res_df['fluxes'] = res_df.reactions.apply(lambda x: sln[tme_new.reactions.index(x)])
res_df.set_index(res_df.reactions, drop = True, inplace = True)
biom = res_df.loc[[i for i in res_df.index if 'biomass' in i],:]

biom.sort_values(by = 'fluxes', ascending = False)

,reactions,fluxes
reactions,,
biomass_dilution,biomass_dilution,1.000000e-09
DNA_biomass_formation,DNA_biomass_formation,1.000000e-09
carbohydrate_biomass_formation,carbohydrate_biomass_formation,1.000000e-09
lipid_biomass_formation,lipid_biomass_formation,1.000000e-09
mRNA_biomass_to_biomass,mRNA_biomass_to_biomass,4.126854e-10
unmodeled_protein_biomass_to_biomass,unmodeled_protein_biomass_to_biomass,2.054807e-10
protein_biomass_to_biomass,protein_biomass_to_biomass,1.997807e-10
lipid_biomass_to_biomass,lipid_biomass_to_biomass,9.700000e-11
carbohydrate_biomass_to_biomass,carbohydrate_biomass_to_biomass,7.100000e-11


In [26]:
import pandas as pd
res_df = pd.DataFrame(data = {'reactions': [r.id for r in tme_new.reactions]})
res_df['fluxes'] = res_df.reactions.apply(lambda x: sln[tme_new.reactions.index(x)])
res_df.set_index(res_df.reactions, drop = True, inplace = True)
biom = res_df.loc[[i for i in res_df.index if 'biomass' in i],:]

biom.sort_values(by = 'fluxes', ascending = False)

,reactions,fluxes
reactions,,
biomass_dilution,biomass_dilution,1.000000e-09
DNA_biomass_formation,DNA_biomass_formation,1.000000e-09
carbohydrate_biomass_formation,carbohydrate_biomass_formation,1.000000e-09
lipid_biomass_formation,lipid_biomass_formation,1.000000e-09
unmodeled_protein_biomass_to_biomass,unmodeled_protein_biomass_to_biomass,8.166589e-10
lipid_biomass_to_biomass,lipid_biomass_to_biomass,9.700000e-11
carbohydrate_biomass_to_biomass,carbohydrate_biomass_to_biomass,7.100000e-11
DNA_biomass_to_biomass,DNA_biomass_to_biomass,1.400000e-11
protein_biomass_to_biomass,protein_biomass_to_biomass,1.153586e-12
